# 08 — Case-control vs one-sample lesion network mapping
Figure 4, Supplementary figures S11 and S14. Requires the renders of 05 (Figure 1, S10).

In [ ]:
import sys; sys.path.insert(0, "utils")
from config import *
from mapstats import *
from figstyle import apply_figure_style, homogenize_fig, render_info, spread_labels, surface_views
from plotting import corr_heatmap
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import Ellipse

brain, affine, _ = load_mask()
degree = np.where(brain, nib.load(DEGREE_MAP).get_fdata(), np.nan)
T_cc = {k: tmap(LNM, k, brain) for k in KEYS}
T_os = {k: tmap(OS, k, brain) for k in KEYS}
MEAN = {k: np.where(brain, nib.load(mean_map(k)).get_fdata(), np.nan) for k in KEYS}
LAB = list(SH.values())
apply_figure_style(sizes=(9, 8, 7))

S11: case-control vs one-sample T maps; S14: one-sample T vs mean aggregation (voxelwise Pearson r).

In [ ]:
M11 = matrix(lambda a, b: pearson(T_cc[a], T_os[b]), KEYS, list(KEYS))
M14 = matrix(lambda a, b: pearson(T_os[a], MEAN[b]), KEYS, list(KEYS))
M11.to_csv(TAB_DIR / "casecontrol_vs_onesample_r.csv"); M14.to_csv(TAB_DIR / "onesample_vs_meanagg_r.csv")
for M, rl, cl, title, fn in [
        (M11, "Case-control T-map", "One-sample T-map",
         "Correspondence between case-control and\none-sample T-maps (voxelwise Pearson r)", "S11_casecontrol_vs_onesample.png"),
        (M14, "One-sample t-test map", "Mean aggregation map",
         "Correspondence between one-sample t-test\nand mean aggregation maps (voxelwise Pearson r)", "S14_onesample_vs_meanagg.png")]:
    fig = corr_heatmap(M.values, LAB, rl, cl, title)
    homogenize_fig(fig); fig.savefig(FIG_DIR / fn, dpi=300, bbox_inches="tight"); plt.close(fig)
print("own-domain r, case-control vs one-sample:", np.round(np.diag(M11), 2))
print("own-domain r, one-sample vs mean:", np.round(np.diag(M14), 2))

Degree alignment (paired t-test on Fisher-z r across domains) and cross-domain similarity.

In [ ]:
r_cc_deg = np.array([pearson(T_cc[k], degree) for k in KEYS])
r_os_deg = np.array([pearson(T_os[k], degree) for k in KEYS])
tt = stats.ttest_rel(np.arctanh(r_os_deg), np.arctanh(r_cc_deg))
iu = np.triu_indices(6, 1)
C_cc = matrix(lambda a, b: 1.0 if a == b else pearson(T_cc[a], T_cc[b]), KEYS).values
C_os = matrix(lambda a, b: 1.0 if a == b else pearson(T_os[a], T_os[b]), KEYS).values
own_ccos = np.diag(M11.values)
print("r with degree, case-control:", r_cc_deg.round(2), "| one-sample:", r_os_deg.round(2),
      f"| paired t(5) = {tt.statistic:.2f}, p = {tt.pvalue:.4f}")
print(f"median cross-domain r: case-control {np.median(C_cc[iu]):.2f}, one-sample {np.median(C_os[iu]):.2f}")
print(f"one-sample: degree r {r_os_deg.mean():.2f} ± {r_os_deg.std():.2f}, cross-domain r {C_os[iu].mean():.2f} ± {C_os[iu].std():.2f}")

# S12: fully adjusted vs cohort-only one-sample maps; S13: mean aggregation maps vs degree
r_full = [pearson(tmap(OS_FULL, k, brain), T_os[k]) for k in KEYS]
r_mean_deg = np.array([pearson(MEAN[k], degree) for k in KEYS])
tt_mean = stats.ttest_rel(np.arctanh(r_mean_deg), np.arctanh(r_cc_deg))
print("r(fully adjusted, cohort-only one-sample):", np.round(r_full, 3))
print(f"mean maps vs degree: r = {r_mean_deg.min():.2f}–{r_mean_deg.max():.2f}; "
      f"paired vs case-control t(5) = {tt_mean.statistic:.2f}, p = {tt_mean.pvalue:.4f}")

Figure 4. Surface views are cropped from the Figure 1 and S10 renders (same colour scales).

In [ ]:
V_cc = surface_views(FIG_DIR / "Fig1_lnm_label_permutation.png")      # {domain: views}; n.s. domains absent
V_os = surface_views(FIG_DIR / "S10_onesample_cohort.png")
VMAX_CC = round(render_info(FIG_DIR / "Fig1_lnm_label_permutation.png")["vmax"])
VMAX_OS = round(render_info(FIG_DIR / "S10_onesample_cohort.png")["vmax"])
apply_figure_style(sizes=(8, 7, 6))
C_CC, C_OS, C_LES, C_LES2 = "#5E3C99", "#E66101", "#018571", "#80CDC1"
DOMLAB = ["Attention /\nexecutive", "Processing\nspeed", "Language", "Verbal\nmemory",
          "Visuospatial\nfunctions", "Visuospatial\nmemory"]

fig = plt.figure(figsize=(7.2, 5.9))
outer = fig.add_gridspec(1, 2, width_ratios=[1.55, 1.0], wspace=0.36, left=0.095, right=0.985, top=0.915, bottom=0.085)
gmap = outer[0].subgridspec(7, 5, width_ratios=[1, 1, 0.12, 1, 1], height_ratios=[1]*6 + [0.32],
                            hspace=0.10, wspace=0.03)
# ---- (a) map block ----
axes_map = []
for r in range(6):
    for c_i, (src, q) in enumerate([("cc", 0), ("cc", 2), (None, None), ("os", 0), ("os", 2)]):
        if src is None: continue
        ax = fig.add_subplot(gmap[r, c_i]); ax.axis("off"); axes_map.append(ax)
        V = V_cc if src == "cc" else V_os
        if KEYS[r] in V: ax.imshow(V[KEYS[r]][q])
        elif q == 0: ax.text(1.0, 0.5, "n.s.", ha="center", va="center", transform=ax.transAxes,
                             fontsize=8, color="0.45")
        if c_i == 0:
            ax.text(-0.08, 0.5, DOMLAB[r], transform=ax.transAxes, ha="right", va="center", fontsize=7)
ax_top_cc = fig.add_subplot(gmap[0, 0:2], frameon=False); ax_top_cc.set_xticks([]); ax_top_cc.set_yticks([])
ax_top_os = fig.add_subplot(gmap[0, 3:5], frameon=False); ax_top_os.set_xticks([]); ax_top_os.set_yticks([])
ax_top_cc.set_title("Label permutation-based\n(case-control)", fontsize=8, color=C_CC, fontweight="bold", pad=4)
ax_top_os.set_title("One-sample t-test\n(impaired patients only)", fontsize=8, color=C_OS, fontweight="bold", pad=4)
for cols, vmax, lab in [((0, 2), VMAX_CC, r"$T$ ($p_{FDR}$ < 0.05)"), ((3, 5), VMAX_OS, r"$T$ ($p_{FDR}$ < 0.05)")]:
    cax_host = fig.add_subplot(gmap[6, cols[0]:cols[1]]); cax_host.axis("off")
    cax = cax_host.inset_axes([0.12, 0.55, 0.76, 0.22])
    cb = fig.colorbar(ScalarMappable(Normalize(-vmax, vmax), "RdBu_r"), cax=cax, orientation="horizontal",
                      extend="both", ticks=[-vmax, 0, vmax])
    cb.ax.tick_params(labelsize=6, length=2)
    for t in cb.ax.get_xticklabels(): t.set_fontweight("bold")
    cb.set_label(lab, fontsize=6, fontweight="bold", labelpad=1)

right = outer[1].subgridspec(2, 1, hspace=0.34)
# ---- (b) degree alignment (S15) ----
ax = fig.add_subplot(right[0])
for a_, b_ in zip(r_cc_deg, r_os_deg):
    ax.plot([0, 1], [a_, b_], color="0.75", lw=0.8, zorder=1)
ax.scatter(np.zeros(6), r_cc_deg, s=22, color=C_CC, zorder=3)
ax.scatter(np.ones(6), r_os_deg, s=22, color=C_OS, zorder=3)
ylab = spread_labels(r_cc_deg, min_gap=0.062)
for k_i, (yv, yl) in enumerate(zip(r_cc_deg, ylab)):
    ax.text(-0.07, yl, SH[KEYS[k_i]], ha="right", va="center", fontsize=5.8, color="0.3")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Case-control", "One-sample"])
for t, c in zip(ax.get_xticklabels(), [C_CC, C_OS]): t.set_color(c)
ax.set_xlim(-0.95, 1.3); ax.set_ylim(-0.3, 1.0); ax.axhline(0, color="0.85", lw=0.6, zorder=0)
ax.set_ylabel("Correlation with\nconnectome degree map (r)")
ax.set_title("Degree-map correlation", loc="left"); ax_b4 = ax
ax.text(0.98, 0.985, f"paired t(5) = {tt.statistic:.2f}, p = {tt.pvalue:.3f}", transform=ax.transAxes,
        ha="right", va="top", fontsize=6)

# ---- (c) similarity strips ----
ax = fig.add_subplot(right[1])
groups = [(C_cc[iu], C_CC, "Case-control,\nacross\ndomains"),
          (C_os[iu], C_OS, "One-sample,\nacross\ndomains"),
          (own_ccos, "0.35", "Case-control vs\none-sample,\nsame domain")]
rng = np.random.default_rng(1)
xpos = []
for x, (vals, col, _) in enumerate(groups):
    xs_ = x + rng.uniform(-0.13, 0.13, len(vals)); xpos.append(xs_)
    ax.scatter(xs_, vals, s=13, color=col, alpha=0.85, zorder=3, lw=0)
    ax.plot([x - 0.24, x + 0.24], [np.median(vals)] * 2, color="black", lw=1.3, zorder=4)
# exemplary dots: one domain pair followed across the two cross-domain strips, one same-domain value
_pl = [(KEYS[i], KEYS[j]) for i, j in zip(*iu)]
k_pair = next(n for n, p in enumerate(_pl) if set(p) == {"cognition-verbalmemory_domain_score", "cognition-vsfunctions_domain_score"})
k_own = KEYS.index("cognition-verbalmemory_domain_score")
EX = [(0, k_pair, "Verbal mem. \u2013\nVS func.", "right"),
      (2, k_own, "Verbal\nmemory", "right")]
ax.set_xticks(range(3)); ax.set_xticklabels([g[2].replace(" across", "\nacross").replace("\nacross\nacross", "\nacross") for g in groups], fontsize=6)
for t, g in zip(ax.get_xticklabels(), groups): t.set_color(g[1])
ax.set_xlim(-0.5, 2.62); ax.set_ylim(-0.35, 1.08); ax.axhline(0, color="0.85", lw=0.6, zorder=0)
fig.canvas.draw()
_bb = ax.get_window_extent(); _dx = 3.12 / _bb.width; _dy = 1.43 / _bb.height     # data units per pixel
R_PX = 9 * fig.dpi / 100                                                         # circle radius in pixels
for g_i, k_, lab_, side in EX:
    xv, yv = xpos[g_i][k_], groups[g_i][0][k_]
    ax.add_patch(Ellipse((xv, yv), 2 * R_PX * _dx, 2 * R_PX * _dy, fill=False, ec="0.55", lw=0.7, zorder=5))
    if side == "right":
        ax.text(xv + R_PX * _dx + 0.04, yv, lab_, ha="left", va="center", fontsize=5.5, color="0.3", linespacing=1.0)
    else:
        ax.text(xv, yv - R_PX * _dy - 0.12, lab_, ha="center", va="top", fontsize=5.5, color="0.3", linespacing=1.0)
ax.set_ylabel("Spatial correlation of\nunthresholded maps (r)")
ax.set_title("Similarity of unthresholded maps", loc="left"); ax_c4 = ax

# ---- letters share the top line of the text they label (all as fixed figure text, layout frozen) ----
fig.canvas.draw()
if fig.get_layout_engine() is not None: fig.set_layout_engine("none")
_r = fig.canvas.get_renderer(); _inv = fig.transFigure.inverted()
_H = fig.get_size_inches()[1] * 72                                  # figure height in pt
def _pop(ax_):
    t = next(t for t in (ax_.title, ax_._left_title, ax_._right_title) if t.get_text())   # whichever slot holds it
    d = dict(text=t.get_text(), fontsize=t.get_fontsize(), color=t.get_color()); t.set_text(""); return d
h_cc, h_os = _pop(ax_top_cc), _pop(ax_top_os)
t_b, t_c = _pop(ax_b4), _pop(ax_c4)
pt = lambda v: v / _H                                               # points -> figure fraction (vertical)
fs_h = h_cc["fontsize"]
b2 = ax_top_cc.get_position().y1 + 0.012 + pt(0.25 * fs_h)          # baseline of 2nd header line
b1 = b2 + pt(1.2 * fs_h)                                            # baseline of 1st header line = shared baseline
y_c = ax_c4.get_position().y1 + 0.022 + pt(0.25 * t_c["fontsize"])  # baseline of panel c title
x_bc = _inv.transform(ax_b4.get_tightbbox(_r).get_points())[0, 0] - 0.012
LG = 0.028
for ax_h, h in ((ax_top_cc, h_cc), (ax_top_os, h_os)):                 # each header line as its own baseline text
    for k_, ln_ in enumerate(h["text"].split("\n")):
        fig.text(ax_h.get_position().x0, b1 - k_ * pt(1.2 * h["fontsize"]), ln_, ha="left", va="baseline",
                 fontsize=h["fontsize"], color=h["color"], fontweight="bold")
fig.text(x_bc + LG, b1, t_b["text"], ha="left", va="baseline", fontsize=t_b["fontsize"], fontweight="bold")
fig.text(x_bc + LG, y_c, t_c["text"], ha="left", va="baseline", fontsize=t_c["fontsize"], fontweight="bold")
x_left = min([_inv.transform(t.get_window_extent(_r).get_points())[0, 0] for t in fig.texts
              if t.get_text().strip() and t.get_position()[0] < 0.2] + [0.02])
for L, x_, y_ in (("a", x_left, b1), ("b", x_bc, b1), ("c", x_bc, y_c)):
    fig.text(x_, y_, L, ha="left", va="baseline", fontsize=10, fontweight="bold")
homogenize_fig(fig)
fig.savefig(FIG_DIR / "Fig4_onesample_vs_casecontrol.png", dpi=300)
fig.savefig(FIG_DIR / "Fig4_onesample_vs_casecontrol.pdf")